# Explainability and Fairness — Exercises

Companion to the note [Explainability and Fairness](Explainability%20and%20Fairness.md).

Practise the explanation methods and fairness criteria from the note: permutation importance, partial dependence, Shapley values (by hand and by exact enumeration), LIME-style local surrogates and counterfactuals; then group fairness metrics, the impossibility result, and pre- and post-processing mitigations (reweighing, group thresholds), with NumPy implementations checked against scikit-learn.

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×5 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from itertools import combinations
from math import factorial
from scipy import stats
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance, partial_dependence
from sklearn.metrics import r2_score
rng = np.random.default_rng(0)

# Shared regression problem: y depends strongly on x0, weakly on x1, not at all on x2
X_reg = rng.normal(size=(1000, 3))
y_reg = 3 * X_reg[:, 0] + 1 * X_reg[:, 1] + 0.5 * rng.normal(size=1000)
X_tr, X_te, y_tr, y_te = X_reg[:700], X_reg[700:], y_reg[:700], y_reg[700:]

## Part A · Concepts

### Exercise 1 · Placing methods on the map
*🧠 Concept · ★☆☆*

For each question say which method from the note's table you would use and whether it is **global** or **local**, **model-specific** or **model-agnostic**:
(a) "Which features matter most for this gradient-boosted model overall?"
(b) "Why was *this* loan application rejected?"
(c) "What would the applicant need to change to be approved?"
(d) "Which pixels made the CNN say 'husky'?"
(e) "How does predicted price change with house size, on average?"

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Global methods summarise the whole model; local methods explain one prediction.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Permutation importance** or mean $|$SHAP$|$: global, model-agnostic (TreeSHAP is the fast model-specific variant).
(b) **SHAP** or **LIME**: local, model-agnostic.
(c) **Counterfactual explanation**: local, model-agnostic (an optimisation over inputs).
(d) **Saliency / Grad-CAM / integrated gradients**: local, model-specific (needs gradients of a [[CNN]]).
(e) **Partial dependence** (global average) with **ICE** curves (one per instance) to reveal heterogeneity.
Intrinsically interpretable models ([[Linear Regression]], small [[Decision Trees]], GAMs) answer (a) and (e) directly from their parameters.

</details>

### Exercise 2 · Where does the bias come from?
*🧠 Concept · ★☆☆*

Name the source of bias from the note's list (historical, sampling, label, proxy, feedback loop) in each case:
(a) A face detector trained mostly on light-skinned faces fails on darker skin.
(b) A hiring model trained on past hiring decisions in a male-dominated field down-ranks women.
(c) Gender is removed from the features, but "attended a women's college" remains.
(d) Predictive policing sends more patrols to a neighbourhood, more arrests are recorded there, and the model becomes more confident.
(e) "Arrested" is used as the label for "committed a crime".

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Each scenario matches exactly one category.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Sampling / representation** bias. (b) **Historical** bias: the data faithfully records past discrimination.
(c) **Proxy**: removing a protected attribute ("fairness through unawareness") does not work when correlated features remain.
(d) **Feedback loop**: the model's actions generate its future training data (also in [[Recommender Systems Overview]]).
(e) **Label (measurement)** bias: the label is a biased proxy for the real target.

</details>

### Exercise 3 · Permutation importance with correlated features
*🧠 Concept · ★★☆*

A model uses two nearly identical features, `height_cm` and `height_in`. Explain what permutation importance reports for each of them and why.
What goes wrong with the "shuffled" data points themselves? Suggest two remedies.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

If you shuffle `height_cm` but keep `height_in`, can the model still recover the information? And is a 190 cm / 60 in person realistic?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Permuting one copy leaves the other intact, so (depending on how the model split the weight) each can look **unimportant**, even though height
as a concept is crucial; importance is shared or hidden. Moreover, shuffling one of two correlated features creates **unrealistic points**
(190 cm with 60 inches) where the model is extrapolating, so the measured drop reflects behaviour off the data manifold.
Remedies: permute **groups** of correlated features together (or cluster features first), drop one copy, or use conditional importance /
SHAP with a dependence-aware background. The same issue affects PDPs (they average over unrealistic combinations; ALE plots fix this).

</details>

### Exercise 4 · LIME vs SHAP
*🧠 Concept · ★★☆*

Compare LIME and (Kernel)SHAP: what each fits or computes, which guarantees SHAP's Shapley values satisfy (name three axioms), and one practical
weakness of each.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

SHAP can be seen as LIME with a specific kernel and loss; Shapley values come from cooperative game theory.

</details>

<details>
<summary><b>✅ Solution</b></summary>

**LIME** samples perturbations around $x$, weights them by proximity, and fits a sparse linear surrogate; its coefficients are the explanation.
**SHAP** attributes $f(x) - \mathbb E[f(X)]$ to features using Shapley values; KernelSHAP estimates them by a weighted linear regression with the
Shapley kernel. Axioms: **efficiency** (attributions sum to $f(x) - \mathbb E f$), **symmetry** (interchangeable features get equal credit),
**dummy/null player** (a feature that never changes the output gets 0), and additivity/linearity.
Weaknesses: LIME's explanation depends on the kernel width and sampling (unstable across runs); SHAP is exponential in exact form (approximations
needed except TreeSHAP), and with correlated features the "interventional" background evaluates unrealistic inputs, much like permutation importance.

</details>

## Part B · By hand

### Exercise 5 · Shapley values for two players
*✍️ By hand · ★☆☆*

A model's value function (expected prediction when only the features in $S$ are known) is $v(\emptyset) = 0$, $v(\{1\}) = 10$, $v(\{2\}) = 20$,
$v(\{1,2\}) = 50$. Compute the Shapley values
$\phi_i = \sum_{S\subseteq N\setminus\{i\}} \frac{|S|!\,(n-|S|-1)!}{n!}\big[v(S\cup\{i\}) - v(S)\big]$ and verify efficiency.

In [ ]:
phi1 = None
phi2 = None

check('phi_1', phi1, 0.5 * ((10 - 0) + (50 - 20)))
check('phi_2', phi2, 0.5 * ((20 - 0) + (50 - 10)))

<details>
<summary><b>💡 Hint</b></summary>

With two players, average the marginal contribution of player $i$ over the two orderings.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\phi_1 = \tfrac12[(10 - 0) + (50 - 20)] = 20$, $\phi_2 = \tfrac12[(20 - 0) + (50 - 10)] = 30$. Efficiency: $20 + 30 = 50 = v(N) - v(\emptyset)$.
The interaction surplus $50 - 10 - 20 = 20$ is split **equally** between the two features.

```python
phi1, phi2 = 20, 30
```

</details>

### Exercise 6 · SHAP for a linear model
*✍️ By hand · ★★☆*

For a linear model $f(x) = b + \sum_j w_j x_j$ with independent features and the interventional value function
$v(S) = \mathbb E[f(x_S, X_{\bar S})]$, show that $\phi_j = w_j(x_j - \mathbb E[X_j])$. Compute $\phi$ for $w = (2, -1)$, $x = (3, 1)$, $\mathbb E[X] = (1, 2)$.

In [ ]:
phi_lin = None   # [phi_1, phi_2]

check('linear SHAP', phi_lin, np.array([2, -1]) * (np.array([3, 1]) - np.array([1, 2])))

<details>
<summary><b>💡 Hint</b></summary>

The marginal contribution of feature $j$ is the same for every coalition $S$, so the weighted average is just that contribution.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Adding feature $j$ to any coalition changes $v$ by $w_j x_j - w_j\mathbb E[X_j]$ (other terms are unaffected), so every marginal contribution is
the same and $\phi_j = w_j(x_j - \mathbb E[X_j])$. Here $\phi = (2\cdot2,\ -1\cdot(-1)) = (4, 1)$, summing to $f(x) - \mathbb E f = 5$.
Note that feature 2 has a negative weight but a **positive** attribution, because this $x_2$ is below average. Attributions are relative to the baseline.

```python
phi_lin = [4, 1]
```

</details>

### Exercise 7 · Group fairness metrics from confusion matrices
*✍️ By hand · ★☆☆*

A classifier's results for two groups of 100 people each:

| Group | TP | FP | FN | TN |
|---|---|---|---|---|
| A | 40 | 10 | 10 | 40 |
| B | 20 | 5 | 20 | 55 |

Compute (a) the demographic-parity difference $|P(\hat y=1\mid A) - P(\hat y=1\mid B)|$ and the disparate-impact ratio (B/A);
(b) the equal-opportunity difference (TPR gap); (c) the equalised-odds difference $\max(|\Delta TPR|, |\Delta FPR|)$; (d) the precision (PPV) of each group.

In [ ]:
dp_diff = None
di_ratio = None
eopp_diff = None
eodds_diff = None

check('DP difference', dp_diff, 50/100 - 25/100)
check('disparate impact', di_ratio, 0.25 / 0.5)
check('equal opportunity diff', eopp_diff, 40/50 - 20/40)
check('equalised odds diff', eodds_diff, max(40/50 - 20/40, 10/50 - 5/60))

<details>
<summary><b>💡 Hint</b></summary>

Selection rate $= (TP+FP)/n$, TPR $= TP/(TP+FN)$, FPR $= FP/(FP+TN)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Selection rates $0.5$ vs $0.25$ → DP difference $0.25$, disparate impact $0.5$ (fails the "80 % rule").
(b) TPR: $40/50 = 0.8$ vs $20/40 = 0.5$ → $0.3$. (c) FPR: $10/50 = 0.2$ vs $5/60 \approx 0.083$ → gap $0.117$; equalised-odds difference $= 0.3$.
(d) PPV: $40/50 = 0.8$ and $20/25 = 0.8$: **equal**. So the classifier satisfies predictive parity (a calibration-type criterion) while violating
demographic parity and equalised odds. The groups have different base rates ($0.5$ vs $0.4$), which is exactly the situation of the next exercise.

```python
dp_diff, di_ratio = 0.25, 0.5
eopp_diff = 0.3
eodds_diff = 0.3
```

</details>

### Exercise 8 · The impossibility result in one formula
*✍️ By hand · ★★★*

Let $p$ be a group's base rate $P(y=1)$. Show that $\text{FPR} = \frac{p}{1-p}\cdot\frac{1-\text{PPV}}{\text{PPV}}\cdot\text{TPR}$ (Chouldechova 2017).
Two groups have base rates $p_A = 0.5$, $p_B = 0.2$, and a classifier has the **same** PPV $= 0.8$ and the same TPR $= 0.7$ in both. Compute both FPRs.
What does this imply about satisfying predictive parity and equalised odds simultaneously?

In [ ]:
fpr_A = None
fpr_B = None

_f = lambda p: p / (1 - p) * (0.2 / 0.8) * 0.7
check('FPR_A', fpr_A, _f(0.5))
check('FPR_B', fpr_B, _f(0.2))

<details>
<summary><b>💡 Hint 1</b></summary>

With $n$ people: $TP = \text{TPR}\cdot p\,n$ and $FP = \text{FPR}\cdot(1-p)\,n$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\frac{1 - \text{PPV}}{\text{PPV}} = \frac{FP}{TP}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac{1-\text{PPV}}{\text{PPV}} = \frac{FP}{TP} = \frac{\text{FPR}(1-p)}{\text{TPR}\,p}$; solve for FPR. With PPV $= 0.8$, TPR $= 0.7$:
$\text{FPR}_A = 1\cdot0.25\cdot0.7 = 0.175$, $\text{FPR}_B = 0.25\cdot0.25\cdot0.7 = 0.04375$.
If base rates differ, equal PPV and equal TPR **force** unequal FPRs, so predictive parity and equalised odds cannot both hold (unless the classifier
is perfect or base rates are equal). This is the note's "generally cannot all hold at once": choosing a fairness criterion is a value judgement about
which errors matter (the COMPAS debate).

```python
fpr_A = 0.175
fpr_B = 0.04375
```

</details>

### Exercise 9 · Counterfactual for a linear classifier
*✍️ By hand · ★★☆*

A linear classifier approves when $w^\top x + b \ge 0$, with $w = (1, 2)$, $b = -4$. The applicant $x = (1, 1)$ is rejected. Find the smallest
change $\delta$ (in Euclidean norm) such that $x + \delta$ lies on the decision boundary, and its length. Why might this "minimal" counterfactual
still be a bad explanation in practice?

In [ ]:
delta_cf = None   # [d1, d2]
dist_cf = None

_w = np.array([1., 2.]); _s = _w @ np.array([1., 1.]) - 4
check('delta', delta_cf, -_s / (_w @ _w) * _w)
check('distance', dist_cf, abs(_s) / np.linalg.norm(_w))

<details>
<summary><b>💡 Hint</b></summary>

The shortest path to a hyperplane is along its normal $w$: $\delta = -\frac{w^\top x + b}{\lVert w\rVert^2}w$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Score $= 1 + 2 - 4 = -1$. $\delta = \frac{1}{5}(1, 2) = (0.2, 0.4)$, length $1/\sqrt5 \approx 0.447$ (the distance to the hyperplane).
In practice the minimal change may be **infeasible or non-actionable** (it may require changing age, or several features in an unrealistic
combination). Good counterfactual methods add constraints: immutable features, plausibility (stay on the data manifold), sparsity (change few
features), and costs per feature.

```python
delta_cf = [0.2, 0.4]
dist_cf = 1 / np.sqrt(5)
```

</details>

## Part C · Code from scratch

### Exercise 10 · Permutation importance
*💻 Code · ★☆☆*

Implement `perm_importance(model, X, y, n_repeats, seed)` returning the mean drop in $R^2$ when each column of the **test** set is shuffled.
Fit a `LinearRegression` on `X_tr, y_tr` and compare with `sklearn.inspection.permutation_importance` on `X_te, y_te`.

In [ ]:
def perm_importance(model, X, y, n_repeats=10, seed=0):
    # TODO: return array of shape (n_features,)
    return None

lin = LinearRegression().fit(X_tr, y_tr)
pi_mine = perm_importance(lin, X_te, y_te, n_repeats=20)

_pi = permutation_importance(lin, X_te, y_te, n_repeats=20, random_state=0).importances_mean
check('importances ≈ sklearn', pi_mine, _pi, tol=0.05)
check('same ranking', None if pi_mine is None else list(np.argsort(pi_mine)), list(np.argsort(_pi)))

<details>
<summary><b>💡 Hint</b></summary>

Baseline score `r2_score(y, model.predict(X))`; for each column and repeat, `Xp = X.copy(); Xp[:, j] = r.permutation(Xp[:, j])`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Feature 0 has a large importance (≈1.8, the $R^2$ collapses), feature 1 about 0.2 and feature 2 ≈ 0. For a linear model with independent
features the importance is about $2w_j^2\operatorname{Var}(x_j)/\operatorname{Var}(y)$ (shuffling doubles the error variance contributed by $x_j$):
$2\cdot9/10.25 \approx 1.76$ and $2/10.25 \approx 0.2$. Always compute it on **held-out** data; on training data it also measures overfitting.

```python
def perm_importance(model, X, y, n_repeats=10, seed=0):
    r = np.random.default_rng(seed)
    base = r2_score(y, model.predict(X))
    imp = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy(); Xp[:, j] = r.permutation(Xp[:, j])
            drops.append(base - r2_score(y, model.predict(Xp)))
        imp[j] = np.mean(drops)
    return imp

lin = LinearRegression().fit(X_tr, y_tr)
pi_mine = perm_importance(lin, X_te, y_te, n_repeats=20)
print(np.round(pi_mine, 3))
```

</details>

### Exercise 11 · Partial dependence and ICE
*💻 Code · ★★☆*

Fit a `RandomForestRegressor(n_estimators=50, random_state=0)` on the non-linear target $y = \sin(2x_0) + x_1^2/2 + \varepsilon$ below.
Implement `ice_curves(model, X, j, grid)` returning a matrix (n_samples × n_grid) of predictions with feature $j$ set to each grid value, and
`pdp = ice.mean(0)`. Compare your PDP for feature 0 with `sklearn.inspection.partial_dependence(..., method='brute', custom_values=...)`.

In [ ]:
X_nl = rng.uniform(-2, 2, size=(400, 2))
y_nl = np.sin(2 * X_nl[:, 0]) + X_nl[:, 1] ** 2 / 2 + 0.1 * rng.normal(size=400)
rf_nl = RandomForestRegressor(n_estimators=50, random_state=0).fit(X_nl, y_nl)
grid0 = np.linspace(-2, 2, 21)

def ice_curves(model, X, j, grid):
    # TODO
    return None

ice0 = ice_curves(rf_nl, X_nl, 0, grid0)
pdp0 = None if ice0 is None else ice0.mean(0)

_pd = partial_dependence(rf_nl, X_nl, [0], method='brute', kind='average', custom_values={0: grid0})
check('PDP matches sklearn', pdp0, _pd['average'][0])

<details>
<summary><b>💡 Hint</b></summary>

For each grid value `g`: `Xg = X.copy(); Xg[:, j] = g; model.predict(Xg)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The PDP for feature 0 traces the sine shape $\sin(2x_0)$ (shifted by the average effect of $x_1$). Here all ICE curves are parallel, because the
target is additive; with interactions (e.g. $y = x_0x_1$) ICE curves would cross and the PDP average could be flat even though the feature
matters a lot, which is why the note pairs PDP with ICE.

```python
X_nl = rng.uniform(-2, 2, size=(400, 2))
y_nl = np.sin(2 * X_nl[:, 0]) + X_nl[:, 1] ** 2 / 2 + 0.1 * rng.normal(size=400)
rf_nl = RandomForestRegressor(n_estimators=50, random_state=0).fit(X_nl, y_nl)
grid0 = np.linspace(-2, 2, 21)

def ice_curves(model, X, j, grid):
    out = np.empty((len(X), len(grid)))
    for k, g in enumerate(grid):
        Xg = X.copy(); Xg[:, j] = g
        out[:, k] = model.predict(Xg)
    return out

ice0 = ice_curves(rf_nl, X_nl, 0, grid0)
pdp0 = ice0.mean(0)
print(np.round(pdp0[::4], 2))
```

</details>

### Exercise 12 · Exact Shapley values by enumeration
*💻 Code · ★★★*

Implement `shapley_exact(f, x, baseline)` for a model `f` (a function of a 2-D array) with the single-baseline value function
$v(S) = f(z)$ where $z_j = x_j$ for $j\in S$ and $z_j = \text{baseline}_j$ otherwise. Enumerate all coalitions with the Shapley weights.

Test it on $f(x) = x_0x_1 + 2x_2$ at $x = (1, 2, 3)$, baseline $0$ (predict the answer by hand first!), then on the random forest from the
previous exercise, checking efficiency $\sum_j\phi_j = f(x) - f(\text{baseline})$.

In [ ]:
def shapley_exact(f, x, baseline):
    # TODO: return array of shape (n_features,)
    return None

f_toy = lambda Z: Z[:, 0] * Z[:, 1] + 2 * Z[:, 2]
phi_toy = shapley_exact(f_toy, np.array([1., 2., 3.]), np.zeros(3))

check('phi for x0*x1 + 2*x2', phi_toy, [1, 1, 6])
if phi_toy is not None and 'rf_nl' in globals():
    _x, _b = X_nl[0], X_nl.mean(0)
    _phi = shapley_exact(lambda Z: rf_nl.predict(Z), _x, _b)
    check('efficiency on the random forest', _phi.sum(), rf_nl.predict(_x[None])[0] - rf_nl.predict(_b[None])[0])

<details>
<summary><b>💡 Hint 1</b></summary>

Weight of coalition $S$ (not containing $j$): $\frac{|S|!(n-|S|-1)!}{n!}$; use `itertools.combinations` over the other features.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

By hand: $x_0x_1 = 2$ is a pure interaction between features 0 and 1, so symmetry splits it equally.

</details>

<details>
<summary><b>✅ Solution</b></summary>

For $x_0x_1 + 2x_2$: the interaction term $2$ is shared equally ($\phi_0 = \phi_1 = 1$), and $\phi_2 = 2\cdot3 = 6$ (as in the linear-model exercise); the sum $8 = f(x) - f(0)$.
Exact enumeration costs $2^{n}$ model evaluations per feature, which is fine for 3 features and hopeless for 100; KernelSHAP samples coalitions, and
TreeSHAP exploits tree structure to compute exact values in polynomial time. Using a single baseline is the simplest choice; SHAP averages over a background
dataset instead.

```python
def shapley_exact(f, x, baseline):
    n = len(x)
    phi = np.zeros(n)
    def v(S):
        z = baseline.copy().astype(float)
        z[list(S)] = x[list(S)]
        return f(z[None])[0]
    for j in range(n):
        others = [i for i in range(n) if i != j]
        for size in range(n):
            for S in combinations(others, size):
                w = factorial(size) * factorial(n - size - 1) / factorial(n)
                phi[j] += w * (v(S + (j,)) - v(S))
    return phi

f_toy = lambda Z: Z[:, 0] * Z[:, 1] + 2 * Z[:, 2]
phi_toy = shapley_exact(f_toy, np.array([1., 2., 3.]), np.zeros(3))
print(phi_toy)
```

</details>

### Exercise 13 · A LIME-style local surrogate
*💻 Code · ★★☆*

The black box is $f(x) = \sin(x_0) + x_1^2$. Explain the prediction at $x^\star = (0, 1)$ the LIME way: sample 5000 perturbations
$x^\star + \mathcal N(0, 0.3^2 I)$, weight them with the kernel $\exp(-\lVert z - x^\star\rVert^2/0.3^2)$, and fit a weighted linear regression.
Store the two surrogate coefficients in `lime_coef`. Compare them with the true local gradient. What happens with a much wider sampling scale (e.g. 3)?

In [ ]:
f_bb = lambda Z: np.sin(Z[:, 0]) + Z[:, 1] ** 2
x_star = np.array([0.0, 1.0])

def lime_explain(f, x, scale=0.3, n=5000, seed=0):
    # TODO: return the surrogate's coefficients (without intercept)
    return None

lime_coef = lime_explain(f_bb, x_star)

check('surrogate ≈ local gradient (cos 0, 2·1)', lime_coef, [np.cos(0.0), 2 * 1.0], tol=0.15)

<details>
<summary><b>💡 Hint</b></summary>

`LinearRegression().fit(Z, f(Z), sample_weight=w).coef_`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The surrogate gives coefficients close to $(1, 2)$, the gradient $\nabla f(x^\star) = (\cos 0, 2\cdot1)$: locally, the black box *is* approximately linear.
With a wide scale the samples see the whole sine wave and parabola, and the coefficients no longer describe the local behaviour (the $x_0$
coefficient shrinks towards the average slope of $\sin$ over a wide range). That sensitivity to the neighbourhood definition is LIME's main weakness.

```python
f_bb = lambda Z: np.sin(Z[:, 0]) + Z[:, 1] ** 2
x_star = np.array([0.0, 1.0])

def lime_explain(f, x, scale=0.3, n=5000, seed=0):
    r = np.random.default_rng(seed)
    Z = x + scale * r.normal(size=(n, len(x)))
    w = np.exp(-np.sum((Z - x) ** 2, axis=1) / scale ** 2)
    return LinearRegression().fit(Z, f(Z), sample_weight=w).coef_

lime_coef = lime_explain(f_bb, x_star)
print(np.round(lime_coef, 3), np.round(lime_explain(f_bb, x_star, scale=3.0), 3))
```

</details>

### Exercise 14 · Fairness metrics in code
*💻 Code · ★☆☆*

Implement `fairness_report(y_true, y_pred, group)` returning a dict with the selection rate, TPR and FPR per group and the four summary numbers
from Part B (`dp_diff`, `di_ratio` = min/max selection rate, `eopp_diff`, `eodds_diff`). The arrays below reconstruct the confusion matrices of
the Part B exercise exactly.

In [ ]:
def make_group(tp, fp, fn, tn):
    y = np.r_[np.ones(tp), np.zeros(fp), np.ones(fn), np.zeros(tn)]
    p = np.r_[np.ones(tp), np.ones(fp), np.zeros(fn), np.zeros(tn)]
    return y, p
yA, pA = make_group(40, 10, 10, 40); yB, pB = make_group(20, 5, 20, 55)
y_f = np.r_[yA, yB]; p_f = np.r_[pA, pB]; g_f = np.array(['A'] * 100 + ['B'] * 100)

def fairness_report(y_true, y_pred, group):
    # TODO
    return None

rep = fairness_report(y_f, p_f, g_f)

_get = lambda k: None if rep is None else rep[k]
check('dp_diff', _get('dp_diff'), 0.25)
check('di_ratio', _get('di_ratio'), 0.5)
check('eopp_diff', _get('eopp_diff'), 0.3)
check('eodds_diff', _get('eodds_diff'), 0.3)

<details>
<summary><b>💡 Hint</b></summary>

Loop over `np.unique(group)`; with masks `m = group == g`: TPR $=$ `y_pred[m & (y_true == 1)].mean()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The numbers match Part B. Libraries such as Fairlearn's `MetricFrame` produce the same per-group table and differences; computing it yourself
makes clear that every criterion is just a conditional rate compared across groups.

```python
def make_group(tp, fp, fn, tn):
    y = np.r_[np.ones(tp), np.zeros(fp), np.ones(fn), np.zeros(tn)]
    p = np.r_[np.ones(tp), np.ones(fp), np.zeros(fn), np.zeros(tn)]
    return y, p
yA, pA = make_group(40, 10, 10, 40); yB, pB = make_group(20, 5, 20, 55)
y_f = np.r_[yA, yB]; p_f = np.r_[pA, pB]; g_f = np.array(['A'] * 100 + ['B'] * 100)

def fairness_report(y_true, y_pred, group):
    rep = {}
    for g in np.unique(group):
        m = group == g
        rep[g] = dict(sel=y_pred[m].mean(), tpr=y_pred[m & (y_true == 1)].mean(), fpr=y_pred[m & (y_true == 0)].mean())
    sel = [r['sel'] for r in rep.values()]; tpr = [r['tpr'] for r in rep.values()]; fpr = [r['fpr'] for r in rep.values()]
    rep['dp_diff'] = max(sel) - min(sel)
    rep['di_ratio'] = min(sel) / max(sel)
    rep['eopp_diff'] = max(tpr) - min(tpr)
    rep['eodds_diff'] = max(rep['eopp_diff'], max(fpr) - min(fpr))
    return rep

rep = fairness_report(y_f, p_f, g_f)
print(rep)
```

</details>

### Exercise 15 · Post-processing: group thresholds for equal opportunity
*💻 Code · ★★★*

A risk score is less informative for group B. With one global threshold of 0, TPRs differ. Keep threshold 0 for group A and choose the
threshold `t_B` for group B so that its TPR matches group A's (equal opportunity). Store `t_B`, the new TPR gap `tpr_gap_after`, and the FPR of
group B before and after (`fpr_B_before`, `fpr_B_after`). What did equal opportunity cost?

In [ ]:
r5 = np.random.default_rng(5)
nA = nB = 5000
yA_s = r5.random(nA) < 0.5; yB_s = r5.random(nB) < 0.5
sA = np.where(yA_s, r5.normal(1.0, 1, nA), r5.normal(-1, 1, nA))
sB = np.where(yB_s, r5.normal(0.5, 1, nB), r5.normal(-1, 1, nB))
tpr_A = np.mean(sA[yA_s] > 0)

t_B = None
tpr_gap_after = None
fpr_B_before = None
fpr_B_after = None

check('TPR gap closed (< 0.02)', None if tpr_gap_after is None else tpr_gap_after < 0.02, True)
check('FPR of B before', fpr_B_before, np.mean(sB[~yB_s] > 0))
check('lower threshold raises FPR_B', None if fpr_B_after is None else fpr_B_after > fpr_B_before, True)

<details>
<summary><b>💡 Hint</b></summary>

The TPR at threshold $t$ is the fraction of B's positive scores above $t$, so pick $t_B$ as the $(1 - \text{TPR}_A)$-quantile of `sB[yB_s]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Group A has TPR $\approx\Phi(1) \approx 0.84$, group B only $\Phi(0.5)\approx 0.69$ at threshold 0. Setting $t_B$ to the $(1-0.84)$-quantile of B's
positive scores (≈ −0.5) equalises the TPRs, but B's FPR rises from ≈0.16 to ≈0.31: more false alarms for group B. Post-processing is cheap and
model-agnostic but requires the group attribute at decision time (often legally sensitive) and cannot fix an uninformative score; better
features or data for group B attack the root cause (Hardt et al. 2016).

```python
r5 = np.random.default_rng(5)
nA = nB = 5000
yA_s = r5.random(nA) < 0.5; yB_s = r5.random(nB) < 0.5
sA = np.where(yA_s, r5.normal(1.0, 1, nA), r5.normal(-1, 1, nA))
sB = np.where(yB_s, r5.normal(0.5, 1, nB), r5.normal(-1, 1, nB))
tpr_A = np.mean(sA[yA_s] > 0)

t_B = np.quantile(sB[yB_s], 1 - tpr_A)
tpr_gap_after = abs(np.mean(sB[yB_s] > t_B) - tpr_A)
fpr_B_before = np.mean(sB[~yB_s] > 0)
fpr_B_after = np.mean(sB[~yB_s] > t_B)
print(round(tpr_A, 3), round(t_B, 3), round(fpr_B_before, 3), round(fpr_B_after, 3))
```

</details>

### Exercise 16 · Pre-processing: reweighing
*💻 Code · ★★☆*

Kamiran & Calders' **reweighing** gives each example the weight $w(g, y) = \frac{P(g)\,P(y)}{P(g, y)}$, so that in the weighted data the label is
independent of the group. Implement `reweigh(group, y)` and verify that the weighted positive rate is the same in both groups. Then train a
logistic regression on the biased data below with and without the weights and store the demographic-parity difference of the predictions
(`dp_plain`, `dp_weighted`).

In [ ]:
r6 = np.random.default_rng(6)
n6 = 4000
g6 = r6.random(n6) < 0.5                       # True = group B
skill = r6.normal(size=n6)
y6 = (skill + 0.8 * (~g6) + 0.5 * r6.normal(size=n6) > 0.5).astype(int)   # historical bias favours group A
X6 = np.c_[skill, g6.astype(float)]            # the group (or a proxy) is available to the model

def reweigh(group, y):
    # TODO: return an array of per-example weights
    return None

w6 = reweigh(g6, y6)
dp_plain = None
dp_weighted = None

if w6 is not None:
    _rate = lambda m: np.sum(w6[m] * y6[m]) / np.sum(w6[m])
    check('weighted positive rates equal', _rate(g6), _rate(~g6))
else:
    check('weighted positive rates equal', None, True)
check('reweighing reduces the DP gap', None if dp_weighted is None else dp_weighted < dp_plain, True)

<details>
<summary><b>💡 Hint</b></summary>

Compute $P(g)$, $P(y)$ and $P(g, y)$ as empirical frequencies of the four (group, label) cells.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Weights are $> 1$ for the under-represented cells (B with $y=1$, A with $y=0$) and $< 1$ for the others. After reweighing, both groups have the same
weighted positive rate (the overall rate), and the classifier trained with `sample_weight` learns a much smaller coefficient on the group feature, so
its demographic-parity gap shrinks substantially. Reweighing changes neither features nor labels, which makes it easy to audit, but it only
targets demographic parity and may reduce accuracy with respect to the (biased) historical labels.

```python
r6 = np.random.default_rng(6)
n6 = 4000
g6 = r6.random(n6) < 0.5
skill = r6.normal(size=n6)
y6 = (skill + 0.8 * (~g6) + 0.5 * r6.normal(size=n6) > 0.5).astype(int)
X6 = np.c_[skill, g6.astype(float)]

def reweigh(group, y):
    w = np.empty(len(y))
    for g in (False, True):
        for lab in (0, 1):
            m = (group == g) & (y == lab)
            w[m] = np.mean(group == g) * np.mean(y == lab) / np.mean(m)
    return w

w6 = reweigh(g6, y6)
dp = lambda pred: abs(pred[g6].mean() - pred[~g6].mean())
dp_plain = dp(LogisticRegression().fit(X6, y6).predict(X6))
dp_weighted = dp(LogisticRegression().fit(X6, y6, sample_weight=w6).predict(X6))
print(round(dp_plain, 3), round(dp_weighted, 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Explainability and Fairness](Explainability%20and%20Fairness.md).*